# Interview Preparation - KnowBe4

## Create fake data set

In [27]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random


# Faker helps generate realistic fake data
fake = Faker()
# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)


# Number of users
n_users = 500
# Create user IDs
user_ids = np.arange(1, n_users + 1)
# Possible countries
countries = [
    "Netherlands",
    "Germany",
    "South Africa",
    "USA",
    "UK",
    "France"
]
# Generate user dataset
users = pd.DataFrame({
    "user_id": user_ids,
    # Random country assignment
    "country": np.random.choice(countries, n_users),
    # Simulate account age in days
    "account_age_days": np.random.randint(1, 3000, n_users)
})
# Display first few rows
users.head()

# Number of events
n_events = 20000
# Event types
event_types = [
    "login",
    "logout",
    "file_download",
    "password_reset",
    "email_click",
    "vpn_access"
]
# Create empty list to store events
event_rows = []
# Generate event rows
for _ in range(n_events):
    # Random user
    user_id = np.random.choice(user_ids)
    # Random timestamp within last 30 days
    timestamp = (
        datetime.now() - timedelta(
            days=np.random.randint(0, 30),
            hours=np.random.randint(0, 24),
            minutes=np.random.randint(0, 60)
        )
    )
    # Random event type
    event_type = np.random.choice(event_types)
    # Generate fake IP address
    ip_address = fake.ipv4_public()
    # Simulate highly imbalanced target
    # About 2% threats
    is_threat = np.random.choice(
        [0, 1],
        p=[0.98, 0.02]
    )
    # Add suspicious behavior patterns
    # Threats more likely to happen at night
    if is_threat == 1:
        timestamp = timestamp.replace(
            hour=np.random.choice([1, 2, 3, 4])
        )
    # Store row
    event_rows.append([
        user_id,
        timestamp,
        event_type,
        ip_address,
        is_threat
    ])
# Create events DataFrame
events = pd.DataFrame(
    event_rows,
    columns=[
        "user_id",
        "timestamp",
        "event_type",
        "ip_address",
        "is_threat"
    ]
)
# Display first rows
events.head()

# Introduce some missing user IDs
missing_indices = np.random.choice(events.index, 100)
events.loc[missing_indices, "user_id"] = np.nan
# Introduce missing IP addresses
missing_ip_indices = np.random.choice(events.index, 50)
events.loc[missing_ip_indices, "ip_address"] = np.nan

# Save datasets
users.to_parquet("users.parquet", index=False)
events.to_parquet("events.parquet", index=False)
print("Parquet files successfully created!")

Parquet files successfully created!


## Load data into DF's

In [28]:
events = pd.read_parquet('events.parquet')
users = pd.read_parquet('users.parquet')

In [29]:
print(events.info())
print("")
print(users.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     19902 non-null  float64       
 1   timestamp   20000 non-null  datetime64[ns]
 2   event_type  20000 non-null  object        
 3   ip_address  19950 non-null  object        
 4   is_threat   20000 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 781.4+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id           500 non-null    int64 
 1   country           500 non-null    object
 2   account_age_days  500 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 11.8+ KB
None


## Join DF's

In [30]:
df = pd.merge(
    events,
    users,
    on='user_id',
    how='left'
)
df

,user_id,timestamp,event_type,ip_address,is_threat,country,account_age_days
0,369.0,2026-04-12 13:20:32.388930,file_download,68.86.218.241,0,Netherlands,2682.0
1,194.0,2026-04-28 02:50:32.391395,logout,207.181.39.96,0,UK,1251.0
2,300.0,2026-05-08 07:32:32.393732,vpn_access,135.9.72.88,0,South Africa,268.0
3,212.0,2026-05-11 17:09:32.394497,password_reset,109.253.55.179,0,Netherlands,1032.0
4,90.0,2026-04-29 07:21:32.394546,vpn_access,205.43.171.97,0,UK,76.0
...,...,...,...,...,...,...,...
19995,480.0,2026-04-26 22:37:32.816584,password_reset,177.17.112.240,0,USA,2398.0
19996,277.0,2026-04-19 10:20:32.816601,file_download,148.61.84.47,0,Netherlands,714.0
19997,22.0,2026-04-17 23:16:32.816617,password_reset,107.191.81.159,0,Netherlands,2208.0
19998,100.0,2026-04-16 15:29:32.816633,vpn_access,194.253.219.28,0,UK,396.0


## Inspect df

In [31]:
df.head(20)

,user_id,timestamp,event_type,ip_address,is_threat,country,account_age_days
0,369.0,2026-04-12 13:20:32.388930,file_download,68.86.218.241,0,Netherlands,2682.0
1,194.0,2026-04-28 02:50:32.391395,logout,207.181.39.96,0,UK,1251.0
2,300.0,2026-05-08 07:32:32.393732,vpn_access,135.9.72.88,0,South Africa,268.0
3,212.0,2026-05-11 17:09:32.394497,password_reset,109.253.55.179,0,Netherlands,1032.0
4,90.0,2026-04-29 07:21:32.394546,vpn_access,205.43.171.97,0,UK,76.0
5,356.0,2026-04-30 02:55:32.394602,login,173.33.42.155,0,France,1995.0
6,10.0,2026-05-07 14:48:32.394642,password_reset,161.242.224.220,0,UK,2490.0
7,168.0,2026-05-10 17:57:32.394682,email_click,213.171.93.1,0,USA,1959.0
8,265.0,2026-05-01 22:34:32.394732,vpn_access,214.254.110.182,0,USA,385.0
9,94.0,2026-04-26 07:58:32.394779,password_reset,189.42.177.168,0,Netherlands,718.0


In [32]:
df['is_threat'].value_counts(normalize=True)

is_threat
0    0.9811
1    0.0189
Name: proportion, dtype: float64

## Clean data
- Important is to signal nulls before filling them
- Nulls in user_id and ip_address might be most interesting

In [33]:
# Signal missing data
df['missing_userid'] = df['user_id'].isna().astype(int)
df['missing_ip'] = df['ip_address'].isna().astype(int)

In [34]:
# Fill nulls in user_id column
df['user_id'] = df['user_id'].fillna(0).astype(int)

# Make sure timestamp is in correct datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Make sure ip address is string and fillna
df['ip_address'] = df['ip_address'].fillna('unknown')
df['ip_address'] = df['ip_address'].astype('string')

# Fill missing country values
df['country'] = df['country'].fillna('unknown')

# Fill missing account age values
df['account_age_days'] = df['account_age_days'].fillna(0)

## Feature Engineering
- Temporarily use timestamp as index for rolling window
- Add features for strange times, like on weekend or at night
- Add time features to check behaviour changes
- Look for behaviour signals in time related events, also in ip_addresses

In [35]:
# Sort values firts
df = df.sort_values(['user_id', 'timestamp'])

# In order to create rolling time window, timestamp needs to be index
df = df.set_index('timestamp')

In [36]:
# Time related features - behaviour
df['is_weekend'] = df.index.day_of_week.isin([5, 6]).astype(int)
df['is_night'] = df.index.hour.isin([0, 1, 2, 3, 4, 5]).astype(int)

# Time related rolling window behaviour
df['user_events_24hrs'] = (
    df.groupby('user_id')['user_id']
      .rolling('24h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

df['user_events_12hrs'] = (
    df.groupby('user_id')['user_id']
      .rolling('12h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

df['user_events_1hr'] = (
    df.groupby('user_id')['user_id']
      .rolling('1h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

In [37]:
# IP related features
# Count each users total IP addresses over a 24hr period
# Factorize IP column first
df['_ip_code'] = pd.factorize(df['ip_address'])[0]

# Check unique IP addresses per user over a 24hr period
df['unique_ips_24hr'] = (
    df.groupby('user_id')['_ip_code']
      .rolling('24h')
      .apply(lambda x: pd.Series(x).nunique(), raw=False) # raw=True would use np.unique() which is faster
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
      .astype(int)
)

df['is_new_ip'] = (
    ~df.groupby('user_id')['ip_address']
      .transform(lambda x: x.duplicated())
).astype(int)

In [38]:
# Encoding countries and event types
# Events
df = pd.get_dummies(df, columns=['event_type'], drop_first=False)

# Countries
df = pd.get_dummies(df, columns=['country'], drop_first=False)

df

,user_id,ip_address,is_threat,account_age_days,missing_userid,missing_ip,is_weekend,is_night,user_events_24hrs,user_events_12hrs,...,event_type_logout,event_type_password_reset,event_type_vpn_access,country_France,country_Germany,country_Netherlands,country_South Africa,country_UK,country_USA,country_unknown
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-04-12 19:54:32.488434,0,200.152.208.84,0,0.0,1,0,1,0,0,0,...,False,True,False,False,False,False,False,False,False,True
2026-04-12 23:22:32.517773,0,179.111.236.190,0,0.0,1,0,1,0,1,1,...,False,False,False,False,False,False,False,False,False,True
2026-04-13 01:00:32.786905,0,27.105.33.167,0,0.0,1,0,0,1,2,2,...,False,False,False,False,False,False,False,False,False,True
2026-04-13 08:03:32.445347,0,40.112.27.247,0,0.0,1,0,0,0,3,3,...,False,False,False,False,False,False,False,False,False,True
2026-04-14 00:03:32.639947,0,167.156.203.44,0,0.0,1,0,0,1,4,3,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-08 15:25:32.497168,500,136.57.88.59,0,118.0,0,0,0,0,3,2,...,False,False,False,False,False,False,False,False,True,False
2026-05-08 23:25:32.781207,500,197.39.124.146,0,118.0,0,0,0,0,3,2,...,False,True,False,False,False,False,False,False,True,False
2026-05-10 04:55:32.652946,500,27.200.89.212,0,118.0,0,0,1,1,4,3,...,False,True,False,False,False,False,False,False,True,False


In [39]:
# Reset index
df = df.reset_index()
df

,timestamp,user_id,ip_address,is_threat,account_age_days,missing_userid,missing_ip,is_weekend,is_night,user_events_24hrs,...,event_type_logout,event_type_password_reset,event_type_vpn_access,country_France,country_Germany,country_Netherlands,country_South Africa,country_UK,country_USA,country_unknown
0,2026-04-12 19:54:32.488434,0,200.152.208.84,0,0.0,1,0,1,0,0,...,False,True,False,False,False,False,False,False,False,True
1,2026-04-12 23:22:32.517773,0,179.111.236.190,0,0.0,1,0,1,0,1,...,False,False,False,False,False,False,False,False,False,True
2,2026-04-13 01:00:32.786905,0,27.105.33.167,0,0.0,1,0,0,1,2,...,False,False,False,False,False,False,False,False,False,True
3,2026-04-13 08:03:32.445347,0,40.112.27.247,0,0.0,1,0,0,0,3,...,False,False,False,False,False,False,False,False,False,True
4,2026-04-14 00:03:32.639947,0,167.156.203.44,0,0.0,1,0,0,1,4,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2026-05-08 15:25:32.497168,500,136.57.88.59,0,118.0,0,0,0,0,3,...,False,False,False,False,False,False,False,False,True,False
19996,2026-05-08 23:25:32.781207,500,197.39.124.146,0,118.0,0,0,0,0,3,...,False,True,False,False,False,False,False,False,True,False
19997,2026-05-10 04:55:32.652946,500,27.200.89.212,0,118.0,0,0,1,1,4,...,False,True,False,False,False,False,False,False,True,False
19998,2026-05-10 07:19:32.407512,500,198.210.20.4,0,118.0,0,0,1,0,1,...,False,True,False,False,False,False,False,False,True,False


## Define features and target

In [52]:
X = df.drop(columns=[
    'timestamp',
    'user_id',
    'ip_address',
    'is_threat'
])

y = df['is_threat']

## Time-based train/test split

In [53]:
# Sort by time before splitting so the test set represents future events
model_df = df.sort_values('timestamp').copy()

target_col = 'is_threat'
drop_cols = [
    'timestamp',
    'user_id',
    'ip_address',
    'is_threat',
    '_ip_code'  # temporary helper used for rolling IP features
]

feature_cols = [col for col in model_df.columns if col not in drop_cols]

X = model_df[feature_cols]
y = model_df[target_col]

split_idx = int(len(model_df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print('Train threat rate:')
print(y_train.value_counts(normalize=True).rename('share'))
print('\nTest threat rate:')
print(y_test.value_counts(normalize=True).rename('share'))


Train rows: 16,000
Test rows: 4,000
Train threat rate:
is_threat
0    0.980563
1    0.019437
Name: share, dtype: float64

Test threat rate:
is_threat
0    0.98325
1    0.01675
Name: share, dtype: float64


## Evaluation helper


In [54]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

model_results = []

def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    print(f'\n{name} - threshold {threshold:.2f}')
    print(classification_report(y_test, y_pred, digits=3, zero_division=0))
    print(pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=['Actual: not threat', 'Actual: threat'],
        columns=['Predicted: not threat', 'Predicted: threat']
    ))

    result = {
        'model': name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_test, y_proba),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
    }

    print(f"PR-AUC: {result['pr_auc']:.3f}")
    print(f"ROC-AUC: {result['roc_auc']:.3f}")

    model_results.append(result)
    return result


## Baseline model - Logistic Regression


In [55]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

log_reg_model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)
evaluate_model('Logistic Regression', log_reg_model, X_test, y_test)



Logistic Regression - threshold 0.50
              precision    recall  f1-score   support

           0      1.000     0.757     0.862      3933
           1      0.066     1.000     0.123        67

    accuracy                          0.761      4000
   macro avg      0.533     0.879     0.492      4000
weighted avg      0.984     0.761     0.849      4000

                    Predicted: not threat  Predicted: threat
Actual: not threat                   2978                955
Actual: threat                          0                 67
PR-AUC: 0.083
ROC-AUC: 0.901


{'model': 'Logistic Regression',
 'threshold': 0.5,
 'accuracy': 0.76125,
 'precision': 0.06555772994129158,
 'recall': 1.0,
 'f1': 0.12304866850321396,
 'pr_auc': 0.08312491685948978,
 'roc_auc': 0.9011331595265473,
 'tp': np.int64(67),
 'fp': np.int64(955),
 'tn': np.int64(2978),
 'fn': np.int64(0)}

## Ensemble model - AdaBoost


In [56]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1,
            class_weight='balanced',
            random_state=42
        ),
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ))
])

ada_model.fit(X_train, y_train)
evaluate_model('AdaBoost', ada_model, X_test, y_test)



AdaBoost - threshold 0.50
              precision    recall  f1-score   support

           0      1.000     0.757     0.862      3933
           1      0.066     1.000     0.123        67

    accuracy                          0.761      4000
   macro avg      0.533     0.879     0.492      4000
weighted avg      0.984     0.761     0.849      4000

                    Predicted: not threat  Predicted: threat
Actual: not threat                   2978                955
Actual: threat                          0                 67
PR-AUC: 0.066
ROC-AUC: 0.879


{'model': 'AdaBoost',
 'threshold': 0.5,
 'accuracy': 0.76125,
 'precision': 0.06555772994129158,
 'recall': 1.0,
 'f1': 0.12304866850321396,
 'pr_auc': 0.06555772994129158,
 'roc_auc': 0.8785914060513602,
 'tp': np.int64(67),
 'fp': np.int64(955),
 'tn': np.int64(2978),
 'fn': np.int64(0)}

## Optional comparison - Random Forest


In [57]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
evaluate_model('Random Forest', rf_model, X_test, y_test)



Random Forest - threshold 0.50
              precision    recall  f1-score   support

           0      1.000     0.760     0.863      3933
           1      0.065     0.985     0.123        67

    accuracy                          0.764      4000
   macro avg      0.533     0.873     0.493      4000
weighted avg      0.984     0.764     0.851      4000

                    Predicted: not threat  Predicted: threat
Actual: not threat                   2989                944
Actual: threat                          1                 66
PR-AUC: 0.063
ROC-AUC: 0.870


{'model': 'Random Forest',
 'threshold': 0.5,
 'accuracy': 0.76375,
 'precision': 0.06534653465346535,
 'recall': 0.9850746268656716,
 'f1': 0.12256267409470752,
 'pr_auc': 0.06330252087887372,
 'roc_auc': 0.8704266615055918,
 'tp': np.int64(66),
 'fp': np.int64(944),
 'tn': np.int64(2989),
 'fn': np.int64(1)}

## Compare models


In [58]:
comparison = (
    pd.DataFrame(model_results)
      .sort_values('pr_auc', ascending=False)
      .reset_index(drop=True)
)

comparison[[
    'model',
    'threshold',
    'precision',
    'recall',
    'f1',
    'pr_auc',
    'roc_auc',
    'tp',
    'fp',
    'fn',
    'tn'
]]


,model,threshold,precision,recall,f1,pr_auc,roc_auc,tp,fp,fn,tn
0,Logistic Regression,0.5,0.065558,1.000000,0.123049,0.083125,0.901133,67,955,0,2978
1,AdaBoost,0.5,0.065558,1.000000,0.123049,0.065558,0.878591,67,955,0,2978
2,Random Forest,0.5,0.065347,0.985075,0.122563,0.063303,0.870427,66,944,1,2989


## Feature importance


In [59]:
ada_importance = (
    pd.Series(
        ada_model.named_steps['classifier'].feature_importances_,
        index=X_train.columns
    )
    .sort_values(ascending=False)
    .head(15)
)

ada_importance


is_night                     1.0
account_age_days             0.0
event_type_login             0.0
country_USA                  0.0
country_UK                   0.0
country_South Africa         0.0
country_Netherlands          0.0
country_Germany              0.0
country_France               0.0
event_type_vpn_access        0.0
event_type_password_reset    0.0
event_type_logout            0.0
event_type_file_download     0.0
missing_userid               0.0
event_type_email_click       0.0
dtype: float64

## Threshold tuning for operational use


In [60]:
from sklearn.metrics import precision_recall_curve

# In threat detection, threshold choice controls analyst workload vs missed threats.
ada_proba = ada_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, ada_proba)

threshold_df = pd.DataFrame({
    'threshold': thresholds,
    'precision': precision[:-1],
    'recall': recall[:-1],
})

threshold_df['f1'] = (
    2 * threshold_df['precision'] * threshold_df['recall']
    / (threshold_df['precision'] + threshold_df['recall']).replace(0, np.nan)
)

threshold_df.sort_values('f1', ascending=False).head(10)


,threshold,precision,recall,f1
1,0.880797,0.065558,1.0,0.123049
0,0.119203,0.016750,1.0,0.032948
